# B5 책 군집과 대표 책

B4에서 선택한 책 벡터에 k-means를 적용합니다. 독자 평점과 최종 평가 구간은 사용하지 않습니다.
군집은 태그 패턴의 묶음이며 실제 장르 정답을 예측한 결과는 아닙니다.


In [1]:
from pathlib import Path
from time import perf_counter
import json
import numpy as np
import pandas as pd
from bookrec.data import load_book_tags, load_books
from bookrec.models.content_based.tuning import ContentBasedConfig, make_content_based
from bookrec.models.content_based.clustering import (
    compare_cluster_counts, choose_cluster_count, cluster_overview, representative_books,
)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
OUT = ROOT / "results" / "content_based"
OUT.mkdir(parents=True, exist_ok=True)
selected = json.loads((OUT / "B4_selected_config.json").read_text())
features = make_content_based(load_book_tags(), ContentBasedConfig(**selected["config"])).features
books = load_books()
SEED, SAMPLE_SIZE, MIN_BOOKS = 42, 2000, 100
K_CANDIDATES = (5, 10, 15, 20)
print("B4 책 특징 설정:", selected["config"])
print("특징 행렬:", features.matrix.shape)


B4 책 특징 설정: {'extended': False, 'weighting': 'binary', 'min_df': 2, 'sublinear_tf': True, 'profile_weighting': 'positive'}
특징 행렬: (10000, 17042)


## 1. 군집 수 비교

후보 k=5·10·15·20, seed=42, k-means++ 초기화, n_init=10, max_iter=300, tol=1e-4, Lloyd 알고리즘입니다.
영벡터 책은 학습에서 빼고 군집 -1로 표시합니다. 모든 후보의 silhouette는 같은 2,000권 표본에서 계산합니다.
Euclidean 거리는 k-means의 최적화 거리와 맞춥니다. inertia는 전체 학습 책에 대한 군집 내 제곱거리 합입니다.
최소 군집 크기 100권 이상인 후보 중 silhouette가 가장 큰 k를 선택하고, 동점이면 작은 k를 선택합니다.
조건을 만족하는 후보가 없으면 중단하며, 결과를 본 뒤 조건을 자동 완화하지 않습니다.


In [2]:
start = perf_counter()
comparison, models = compare_cluster_counts(features, K_CANDIDATES, seed=SEED, sample_size=SAMPLE_SIZE)
comparison["eligible"] = comparison["min_books"] >= MIN_BOOKS
comparison.to_csv(OUT / "B5_cluster_comparison.csv", index=False)
best_k = choose_cluster_count(comparison, min_books=MIN_BOOKS)
clusters = models[best_k]
print(comparison.to_string(index=False))
print(f"선택 k={best_k}; 비교 실행 {perf_counter() - start:.2f}초")


 k     inertia  silhouette  sample_books  min_books  max_books  unassigned_books  n_iter  eligible
 5 7425.748010    0.052822          2000       1100       3920                 0      12      True
10 6885.840286    0.062676          2000        393       1546                 0      17      True
15 6630.583214    0.056178          2000        385        990                 0      28      True
20 6458.223473    0.053485          2000        164        981                 0      40      True
선택 k=10; 비교 실행 4.70초


## 2. 군집별 대표 태그와 책

top_tags는 중심의 가중치가 높은 태그, distinctive_tags는 군집 중심 가중치에서 전체 책 평균을 뺀 값이 높은 태그입니다.
대표 책은 군집 내부에서 중심까지 Euclidean 거리가 가까운 순서로 고르며, 동점은 작은 book_id가 먼저입니다.
군집 이름은 붙이지 않고 태그와 책 제목을 함께 해석합니다. basic 정제에서 남은 개인 평가·독서 연도 태그도 나타날 수 있습니다.


In [3]:
overview = cluster_overview(features, clusters, n_tags=8)
representatives = representative_books(features, clusters, n_books=5)
representatives = representatives.merge(books[["book_id", "title", "main_author"]], on="book_id", how="left", validate="many_to_one")
assignments = pd.DataFrame({"book_id": clusters.book_ids, "cluster": clusters.labels})
overview.to_csv(OUT / "B5_cluster_overview.csv", index=False)
representatives.to_csv(OUT / "B5_representative_books.csv", index=False)
assignments.to_csv(OUT / "B5_assignments.csv", index=False)
print(overview.to_string(index=False))
print(representatives.to_string(index=False))
selection = {
    "k": best_k, "seed": SEED, "n_init": 10, "max_iter": 300, "tol": 1e-4, "algorithm": "lloyd",
    "initialization": "k-means++", "silhouette_metric": "euclidean", "sample_size": SAMPLE_SIZE,
    "minimum_cluster_books": MIN_BOOKS,
    "criterion": "highest sampled silhouette among candidates with min_books>=100; tie: smaller k",
    "feature_candidate_id": selected["candidate_id"], "feature_config": selected["config"],
    "silhouette": float(comparison.loc[comparison["k"] == best_k, "silhouette"].iloc[0]),
    "unassigned_books": int((clusters.labels < 0).sum()),
}
(OUT / "B5_selected_clustering.json").write_text(json.dumps(selection, ensure_ascii=False, indent=2) + "\n")


 cluster  books                                                                                                   top_tags                                                                                                                                distinctive_tags
       0   1301                          fiction | favorites | literature | novels | favourites | classics | novel | adult                             literature | classics | literary-fiction | classic | literary | general-fiction | historical-fiction | 20th-century
       1    393                 favorites | fiction | paranormal | fantasy | series | adult | supernatural | urban-fantasy                       paranormal-romance | paranormal-fantasy | urban-fantasy | paranormal | supernatural | pnr | fantasy-paranormal | vampires
       2   1385          fiction | mystery | favorites | mystery-thriller | thriller | suspense | crime | mystery-suspense                                             mystery-thriller | mystery-suspe

## 3. ID 대응과 거리 검증

모든 책 ID가 한 번씩 저장되고 대표 책이 해당 군집에 속하며 중심과의 거리 순서로 나오는지 확인합니다.


In [4]:
assert len(assignments) == len(features.book_ids) == 10000
assert assignments["book_id"].is_unique
assert np.array_equal(assignments["book_id"], features.book_ids)
assert overview["books"].sum() + (assignments["cluster"] < 0).sum() == len(assignments)
assert representatives["book_id"].is_unique and representatives["title"].notna().all()
assert np.isfinite(representatives["distance"]).all()
labels = assignments.set_index("book_id")["cluster"]
for cluster, group in representatives.groupby("cluster"):
    assert labels.loc[group["book_id"]].eq(cluster).all()
    assert group["distance"].is_monotonic_increasing
    assert len(group) == 5
print(f"전체 {len(assignments):,}권의 군집 대응과 대표 책 {len(representatives)}권 검증 통과")


전체 10,000권의 군집 대응과 대표 책 50권 검증 통과


## 4. 발표용 그래프

공통 setup·모델 색을 사용하며 저장은 bookrec.plot_style.save()에 맡깁니다.


In [5]:
import os
import tempfile
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "bookrec-matplotlib"))
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from bookrec.plot_style import setup, save, MODEL_COLORS

setup()
fig, axes = plt.subplots(1, 2)
axes[0].plot(comparison["k"], comparison["inertia"], marker="o", color=MODEL_COLORS["cb"])
axes[0].set_ylabel("Inertia (all nonzero books)")
axes[1].plot(comparison["k"], comparison["silhouette"], marker="o", color=MODEL_COLORS["cb"])
axes[1].scatter([best_k], [selection["silhouette"]], color=MODEL_COLORS["mf"], s=70, zorder=3, label=f"Selected k={best_k}")
axes[1].set_ylabel("Silhouette (fixed sample)")
axes[1].legend()
for ax in axes:
    ax.set_xticks(K_CANDIDATES)
    ax.set_xlabel("Number of clusters (k)")
fig.suptitle("B5: choosing the number of book clusters")
save(fig, "B5_cluster_count_comparison")
plt.close(fig)
fig, ax = plt.subplots()
ax.bar(overview["cluster"].astype(str), overview["books"], color=MODEL_COLORS["cb"])
ax.set_xlabel("Cluster ID (not a genre label)")
ax.set_ylabel("Books")
ax.set_title(f"B5: cluster sizes (k={best_k})")
save(fig, "B5_cluster_sizes")
plt.close(fig)
print("공용 save로 그래프 2개 저장 (1600 x 900)")


공용 save로 그래프 2개 저장 (1600 x 900)


## 활용과 한계

신규 독자의 초기 취향 선택 화면에서 각 군집 대표 책을 보여 줄 수 있습니다. 화면 자체는 이번 구현 범위에 포함하지 않습니다.
k-means의 군집 번호는 임의이며 실제 장르 정답과 동일하다고 가정하지 않습니다. 같은 시리즈가 대표 책에 여러 권 포함될 수 있습니다.
silhouette는 고정 표본 2,000권의 내부 지표이고 추천 정확도나 최종 평가 점수를 뜻하지 않습니다.
